# Baby Food Science — Import Pipeline

**Steps:**
1. Fetch OpenAlex counts for all 100 topics (determines node sizes in the map)
2. Browse the 100 available topics and pick which ones to import
3. Run the import (fetches papers from OpenAlex into SQLite)
4. Build the frontend JSON files (`universe.json` + per-topic `nodes/edges.json`)
5. Inspect what was imported

AI enrichment is a separate optional step — skip it to get a working map immediately.

In [1]:
import sys, os, glob, sqlite3

BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR    = os.path.abspath(os.path.join(BACKEND_DIR, "..", "data"))
OUT_DIR     = os.path.abspath(os.path.join(BACKEND_DIR, "..", "frontend", "public"))

sys.path.insert(0, BACKEND_DIR)
from import_openalex import PREDEFINED_TOPICS, import_topic, fetch_all_topic_counts
from build_data import build_universe

print(f"Backend : {BACKEND_DIR}")
print(f"Data    : {DATA_DIR}")
print(f"Output  : {OUT_DIR}")
print(f"Topics available: {len(PREDEFINED_TOPICS)}")

Backend : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\backend
Data    : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\data
Output  : C:\Users\TomHi\Documents\GitHub\mapo_baby_food\frontend\public
Topics available: 100


## Step 1 — Fetch OpenAlex paper counts

Makes one lightweight API call per topic (no paper download) and saves counts to
`data/topic_counts.json`. These totals drive node size in the galaxy map.

Run this once upfront; re-run anytime to refresh the counts.

In [4]:
counts = fetch_all_topic_counts(out_path=os.path.join(DATA_DIR, "topic_counts.json"))
print(f"Fetched counts for {len(counts)} topics.")
top10 = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 by total OpenAlex papers:")
for k, v in top10:
    print(f"  {k:45s}  {v:>8,d}")

  [  1/100] peanut_allergy                                       0
  [  2/100] egg_allergy                                        281
  [  3/100] cow_milk_allergy                                   188
  [  4/100] tree_nut_allergy                                    98
  [  5/100] wheat_gluten_allergy                                22
  [  6/100] soy_allergy                                        163
  [  7/100] sesame_allergy                                     102
  [  8/100] fish_shellfish_allergy                             321
  [  9/100] multiple_food_allergy                                8
  [ 10/100] oral_immunotherapy_allergy                         115
  [ 11/100] early_allergen_introduction                         36
  [ 12/100] allergy_prevention_diet                          1,784
  [ 13/100] eczema_food_allergy                                323
  [ 14/100] food_allergy_anaphylaxis                           536
  [ 15/100] complementary_feeding                            2

## Step 2 — Browse topics

All 100 topics grouped by category. Topics that already have a local database show their current paper count.

In [5]:
def db_paper_count(topic_key):
    path = os.path.join(DATA_DIR, f'papers_{topic_key}.db')
    if not os.path.exists(path):
        return None
    try:
        conn = sqlite3.connect(path)
        n = conn.execute('SELECT COUNT(*) FROM papers').fetchone()[0]
        conn.close()
        return n
    except Exception:
        return 0

# Group topics by their section (inferred from TOPIC_GROUP_MAP order)
GROUPS = [
    ('Allergens',      ['peanut_allergy','egg_allergy','cow_milk_allergy','tree_nut_allergy',
                        'wheat_gluten_allergy','soy_allergy','sesame_allergy','fish_shellfish_allergy',
                        'multiple_food_allergy','oral_immunotherapy_allergy','early_allergen_introduction',
                        'allergy_prevention_diet','eczema_food_allergy','food_allergy_anaphylaxis']),
    ('Feeding Methods',['complementary_feeding','breastfeeding','infant_formula','baby_led_weaning',
                        'responsive_feeding','donor_breast_milk','mixed_feeding',
                        'formula_preparation_safety','extended_breastfeeding',
                        'breastfeeding_difficulties','preterm_infant_nutrition']),
    ('Nutrients',      ['iron_deficiency','vitamin_d','omega3_dha','zinc_infant','calcium_bone_infant',
                        'iodine_infant','folate_infant','vitamin_a_infant','vitamin_b12_infant',
                        'vitamin_k_infant','choline_infant','probiotics_infant','prebiotics_infant']),
    ('Specific Foods', ['vegetable_introduction','fruit_introduction','meat_introduction',
                        'fish_introduction','dairy_introduction','legume_introduction',
                        'whole_grain_infant','organic_baby_food','sugar_salt_babies',
                        'ultra_processed_infant','commercial_baby_food','plant_based_infant']),
    ('Gut Health',     ['gut_microbiome','infant_colic','infant_constipation','infant_reflux',
                        'gut_dysbiosis_infant','food_texture_progression']),
    ('Growth & Dev',   ['infant_growth_faltering','childhood_obesity_diet','stunting_wasting',
                        'brain_development_nutrition','catch_up_growth','dental_health_infant',
                        'toddler_milk_drinks']),
    ('Feeding Behaviour',['food_neophobia','picky_eating','feeding_difficulties','appetite_regulation',
                          'flavor_learning','mealtime_behaviour','division_of_responsibility']),
    ('Maternal',       ['maternal_diet_breastmilk','maternal_nutrition_pregnancy',
                        'prenatal_allergy_prevention','gestational_diabetes_feeding',
                        'maternal_microbiome','prenatal_omega3','maternal_iodine_pregnancy',
                        'maternal_anaemia_infant']),
    ('Special Populations',['low_birth_weight_feeding','celiac_disease_infant','fpies',
                            'eosinophilic_esophagitis','cleft_palate_feeding','downs_syndrome_feeding',
                            'immune_development_diet','fortified_complementary_foods']),
    ('Food Safety',    ['heavy_metals_baby_food','pesticides_infant_food','nitrates_baby_food',
                        'microplastics_formula','bpa_packaging_infant','food_safety_preparation']),
    ('Socioeconomic',  ['food_insecurity_infant','cultural_complementary_feeding',
                        'global_malnutrition_infant','baby_food_marketing','baby_food_labelling',
                        'socioeconomic_infant_diet','sleep_feeding_infant','screen_time_feeding']),
]

total_imported = 0
for group_name, keys in GROUPS:
    print(f'\nâ”€â”€ {group_name} â”€â”€')
    for k in keys:
        cfg = PREDEFINED_TOPICS[k]
        count = db_paper_count(k)
        status = f'{count:4d} papers' if count is not None else '   (not imported)'
        print(f'  {k:40s}  {cfg["name"]:45s}  {status}')
        if count:
            total_imported += count

existing = len(glob.glob(os.path.join(DATA_DIR, 'papers_*.db')))
print(f'\n{existing} databases on disk, {total_imported:,} papers total')


â”€â”€ Allergens â”€â”€
  peanut_allergy                            Peanut Allergy                                    0 papers
  egg_allergy                               Egg Allergy                                     200 papers
  cow_milk_allergy                          Cow Milk Allergy                                188 papers
  tree_nut_allergy                          Tree Nut Allergy                                 98 papers
  wheat_gluten_allergy                      Wheat & Gluten Allergy                           22 papers
  soy_allergy                               Soy Allergy                                     163 papers
  sesame_allergy                            Sesame Allergy                                  102 papers
  fish_shellfish_allergy                    Fish & Shellfish Allergy                        200 papers
  multiple_food_allergy                     Multiple Food Allergies                           8 papers
  oral_immunotherapy_allergy                Oral

## Step 3 â€” Configure & import

Edit `TOPICS_TO_IMPORT` to select which topics to fetch.  
Use `list(PREDEFINED_TOPICS.keys())` to import all 100.

`MAX_PAPERS` controls how many papers to fetch per topic from OpenAlex.  
`MIN_CITATIONS` filters out papers with fewer citations (0 = include everything).

In [7]:
# â”€â”€ configure here â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# TOPICS_TO_IMPORT = [
#     'peanut_allergy',
#     'breastfeeding',
#     'complementary_feeding',
#     'iron_deficiency',
#     'gut_microbiome',
# ]

# To import all 100 topics uncomment this:
TOPICS_TO_IMPORT = list(PREDEFINED_TOPICS.keys())

MAX_PAPERS    = 200   # papers per topic
MIN_CITATIONS = 0     # set higher (e.g. 5) to skip low-impact papers
SKIP_EXISTING = True  # skip topics that already have a database

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

os.makedirs(DATA_DIR, exist_ok=True)

to_run = []
for key in TOPICS_TO_IMPORT:
    if key not in PREDEFINED_TOPICS:
        print(f'  [warn] unknown topic key: {key} â€” skipping')
        continue
    existing_count = db_paper_count(key)
    if SKIP_EXISTING and existing_count is not None:
        print(f'  [skip] {key} â€” already has {existing_count} papers')
        continue
    to_run.append(key)

print(f'\nWill import {len(to_run)} topic(s): {", ".join(to_run)}')

  [skip] peanut_allergy â€” already has 0 papers
  [skip] egg_allergy â€” already has 200 papers
  [skip] cow_milk_allergy â€” already has 188 papers
  [skip] tree_nut_allergy â€” already has 98 papers
  [skip] wheat_gluten_allergy â€” already has 22 papers
  [skip] soy_allergy â€” already has 163 papers
  [skip] sesame_allergy â€” already has 102 papers
  [skip] fish_shellfish_allergy â€” already has 200 papers
  [skip] multiple_food_allergy â€” already has 8 papers
  [skip] oral_immunotherapy_allergy â€” already has 115 papers
  [skip] early_allergen_introduction â€” already has 36 papers
  [skip] allergy_prevention_diet â€” already has 200 papers
  [skip] eczema_food_allergy â€” already has 200 papers
  [skip] food_allergy_anaphylaxis â€” already has 200 papers
  [skip] complementary_feeding â€” already has 200 papers
  [skip] breastfeeding â€” already has 219 papers
  [skip] infant_formula â€” already has 200 papers
  [skip] baby_led_weaning â€” already has 200 papers
  [skip] resp

In [8]:
results = {}
for i, key in enumerate(to_run, 1):
    cfg = PREDEFINED_TOPICS[key]
    print(f'\n[{i}/{len(to_run)}] {cfg["name"]} ({key})')
    print(f'  Query: "{cfg["query"]}"')
    try:
        db_path = import_topic(
            key,
            max_results=MAX_PAPERS,
            min_citations=MIN_CITATIONS,
        )
        count = db_paper_count(key) or 0
        results[key] = count
        print(f'  â†’ {count} papers in database')
    except Exception as e:
        print(f'  [error] {e}')
        results[key] = 0

print(f'\nDone. {sum(results.values()):,} papers imported across {len(results)} topics.')


Done. 0 papers imported across 0 topics.


## Step 4 â€” Build frontend JSON

Reads all databases in `data/` and writes:
- `frontend/public/universe.json` â€” the top-level galaxy map
- `frontend/public/data/<topic>/nodes.json`
- `frontend/public/data/<topic>/edges.json`

Refresh the browser after this runs.

In [9]:
os.makedirs(OUT_DIR, exist_ok=True)
build_universe(DATA_DIR, OUT_DIR)
print('\nFrontend JSON built. Refresh http://localhost:3000 to see the updated map.')

[build] Found 100 databases
[build] Processing allergy_prevention_diet...
  [ok] allergy_prevention_diet: 200 papers, 103 citations
[build] Processing appetite_regulation...
  [ok] appetite_regulation: 30 papers, 10 citations
[build] Processing baby_food_labelling...
  [ok] baby_food_labelling: 200 papers, 314 citations
[build] Processing baby_food_marketing...
  [ok] baby_food_marketing: 200 papers, 351 citations
[build] Processing baby_led_weaning...
  [ok] baby_led_weaning: 200 papers, 386 citations
[build] Processing bpa_packaging_infant...
  [ok] bpa_packaging_infant: 162 papers, 14 citations
[build] Processing brain_development_nutrition...
  [ok] brain_development_nutrition: 78 papers, 16 citations
[build] Processing breastfeeding...
  [ok] breastfeeding: 219 papers, 37 citations
[build] Processing breastfeeding_difficulties...
  [ok] breastfeeding_difficulties: 159 papers, 42 citations
[build] Processing calcium_bone_infant...
  [ok] calcium_bone_infant: 200 papers, 66 citation

## Step 5 â€” Inspect what was imported

In [10]:
dbs = sorted(glob.glob(os.path.join(DATA_DIR, 'papers_*.db')))
if not dbs:
    print('No databases found. Run Step 2 first.')
else:
    total = 0
    print(f'{"Topic":45s} {"Papers":>8s} {"Citations (max)":>16s} {"Year range":>12s}')
    print('-' * 85)
    for db_path in dbs:
        key = os.path.basename(db_path).replace('papers_', '').replace('.db', '')
        conn = sqlite3.connect(db_path)
        try:
            rows = conn.execute(
                'SELECT COUNT(*) as n, MAX(cited_by_count) as max_cites, '
                'MIN(year) as yr_min, MAX(year) as yr_max FROM papers'
            ).fetchone()
            n, max_c, yr0, yr1 = rows
            name = PREDEFINED_TOPICS.get(key, {}).get('name', key)
            yr_range = f'{yr0}â€“{yr1}' if yr0 and yr1 else 'unknown'
            print(f'{name:45s} {n:>8,d} {(max_c or 0):>16,d} {yr_range:>12s}')
            total += n
        except Exception as e:
            print(f'{key}: error â€” {e}')
        finally:
            conn.close()
    print('-' * 85)
    print(f'{"TOTAL":45s} {total:>8,d}')

Topic                                           Papers  Citations (max)   Year range
-------------------------------------------------------------------------------------
Dietary Allergy Prevention                         200            2,790  1999â€“2025
Appetite Regulation in Infants                      30              139  2002â€“2026
Baby Food Labelling & Regulation                   200              233  1994â€“2026
Baby Food Marketing & Industry                     200              339  2001â€“2025
Baby-Led Weaning                                   200            1,119  1930â€“2024
BPA & Food Packaging                               162              191  2005â€“2026
Brain & Cognitive Development                       78              879  2009â€“2026
Breastfeeding                                      219           27,333  1985â€“2025
Breastfeeding Difficulties                         159              228  1999â€“2026
Calcium & Bone Development                         200          

## Optional â€” AI enrichment

Run this later to add recommendation summaries, evidence strength ratings, and likelihood scores.  
Requires Ollama running locally: `ollama serve && ollama pull mistral`

After running, re-run Step 3 to rebuild the frontend JSON with enriched data.

In [ ]:
# from process_ai import process_db, get_client
#
# client, model = get_client(model='mistral')
# for db_path in sorted(glob.glob(os.path.join(DATA_DIR, 'papers_*.db'))):
#     print(f'Enriching {os.path.basename(db_path)}...')
#     process_db(db_path, client, model, batch_size=10)
#
# # Then rebuild:
# build_universe(DATA_DIR, OUT_DIR)